# Bradford Bulls — Logo Detector (RF-DETR) trên Google Colab

So sánh công bằng với notebook YOLO: **cùng dataset, cùng split clip-aware, cùng 17 brand**.

Pipeline:
1. Cài `rfdetr[plus]` + `supervision` + `wandb`.
2. (Tùy chọn) Mount Drive + đăng nhập wandb.
3. Tải dataset **COCO** từ Roboflow (bạn thay link curl COCO của mình).
4. Convert **COCO → COCO**: gộp 32→17 brand + re-split clip-aware ~18%.
5. Train **RF-DETR Large** (704, Apache) — hoặc **2XLarge** (880, PML) cho capacity tối đa.

> **Runtime → GPU (A100/H100)**. Dùng **resolution mặc định** của từng biến thể, không override.

## 0. Cài thư viện + GPU

In [ ]:
!nvidia-smi
# [train,loggers] = thu vien training (pytorch_lightning...) BAT BUOC de goi .train()
!pip -q install -U "rfdetr[train,loggers]" supervision wandb
# Chi can them dong duoi neu chay Option B (XLarge/2XLarge):
# !pip -q install rfdetr-plus
import importlib.metadata as im
def _ver(p):
    try: return im.version(p)
    except Exception: return '?'
import rfdetr, supervision, wandb   # kiem tra import duoc
print('rfdetr', _ver('rfdetr'), '| supervision', _ver('supervision'), '| wandb', _ver('wandb'))
# >>> Sau khi cai lan dau, nen Runtime -> Restart session roi chay lai tu day. <<<

In [ ]:
# (Tùy chọn) Mount Drive để checkpoint không mất khi Colab ngắt phiên
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/bradford_logo_runs'
else:
    PROJECT = '/content/runs'
import os; os.makedirs(PROJECT, exist_ok=True)
print('Runs ->', PROJECT)

In [ ]:
# Đăng nhập wandb (nhập 2 = tài khoản đã có, rồi dán API key từ https://wandb.ai/authorize)
import wandb
wandb.login()

## 1. Tải dataset COCO từ Roboflow (thay link curl COCO của bạn)

In [ ]:
# >>> THAY link curl bang link export dinh dang COCO cua ban <<<
# (Roboflow -> Download -> format COCO -> copy link curl)
!rm -rf /content/roboflow && mkdir -p /content/roboflow
!curl -L "PASTE_COCO_CURL_LINK_HERE" -o /content/roboflow.zip
!unzip -q -o /content/roboflow.zip -d /content/roboflow && rm /content/roboflow.zip

EXPORT_DIR = '/content/roboflow'   # phai chua train/valid/test, moi cai 1 _annotations.coco.json
import os, glob
print('Nội dung:', os.listdir(EXPORT_DIR))
assert glob.glob(EXPORT_DIR + '/*/_annotations.coco.json'), \
    'Khong thay _annotations.coco.json trong cac split — link phai la dinh dang COCO.'

## 2. Convert COCO → COCO (merge 17 brand + split clip-aware)

Gom toàn bộ train+valid+test, gộp `*_home/*_away` → 17 brand, chia lại theo **cụm trận/clip** (không rò rỉ frame ~2 fps). Ghi `train/`, `valid/`, `test/` mỗi cái một `_annotations.coco.json`. (`test` = bản sao `valid` để thỏa loader.)

> Dùng **cùng logic split** với notebook YOLO → cùng tập val → so sánh công bằng.

In [ ]:
import json, re, shutil, random
from collections import Counter, defaultdict
from pathlib import Path

OUT      = '/content/data_coco'
VAL_FRAC = 0.18

BRAND_ORDER = ['acs_group','aon','atm','bartercard','cch','chadlaw','ellgren',
    'em_workwear','fairway','floor_tonic','klg','mcp','mna_cladding',
    'mna_support_service','paints_lacquers','romantica','top_notch']
BRAND_IDX = {b: i for i, b in enumerate(BRAND_ORDER)}

def brand_of(name):
    return re.sub(r'_(home|away)$', '', name)

def group_key(stem):
    m = re.match(r'^(M\d+)', stem)
    if m: return m.group(1)
    m = re.match(r'^(clip_\d+)', stem)
    if m: return m.group(1)
    return stem

# 1) gom moi anh tu tat ca cac split COCO (kem bbox da remap ve brand)
all_imgs = []
for jpath in sorted(Path(EXPORT_DIR).glob('*/_annotations.coco.json')):
    sp_dir = jpath.parent
    j = json.loads(jpath.read_text())
    cat_to_brand = {c['id']: BRAND_IDX.get(brand_of(c['name'])) for c in j['categories']}
    anns_by = defaultdict(list)
    for a in j['annotations']:
        b = cat_to_brand.get(a['category_id'])
        if b is not None:
            anns_by[a['image_id']].append((b, a['bbox']))
    for im in j['images']:
        all_imgs.append({'src': sp_dir/im['file_name'], 'fn': im['file_name'],
                         'w': im['width'], 'h': im['height'],
                         'anns': anns_by.get(im['id'], [])})
print('Tong so anh:', len(all_imgs))

# 2) gom theo cum + dem class
groups = defaultdict(lambda: {'items': [], 'counts': Counter()})
for im in all_imgs:
    g = groups[group_key(Path(im['fn']).stem)]
    g['items'].append(im)
    for b, _ in im['anns']: g['counts'][b] += 1
gkeys = sorted(groups); total = len(all_imgs)
totals = Counter()
for g in groups.values(): totals.update(g['counts'])

# 3) split clip-aware ~VAL_FRAC, uu tien giu class hiem o train
def one_split(seed):
    order = list(gkeys); random.Random(seed).shuffle(order)
    val, n = set(), 0
    for k in order:
        if n >= VAL_FRAC*total: break
        val.add(k); n += len(groups[k]['items'])
    return val
def score(valset):
    tr, va = Counter(), Counter()
    for k, g in groups.items():
        (va if k in valset else tr).update(g['counts'])
    bad = sum(1 for c in totals if tr[c] < 0.6*totals[c])
    nval = sum(len(groups[k]['items']) for k in valset)
    return (bad, abs(nval/(total or 1) - VAL_FRAC))
best = min((one_split(s) for s in range(300)), key=score)

train_items = [it for k, g in groups.items() if k not in best for it in g['items']]
val_items   = [it for k, g in groups.items() if k in best     for it in g['items']]

# 4) ghi COCO (id 0 = placeholder, 1..17 = brand)
categories = [{'id': 0, 'name': 'logos', 'supercategory': 'none'}] + \
    [{'id': i+1, 'name': b, 'supercategory': 'logos'} for i, b in enumerate(BRAND_ORDER)]

def build_split(items, name):
    d = {'images': [], 'annotations': [], 'categories': categories}
    img_dir = Path(OUT)/name; img_dir.mkdir(parents=True, exist_ok=True)
    ann_id = 1
    for img_id, im in enumerate(items, 1):
        shutil.copy2(im['src'], img_dir/im['fn'])
        d['images'].append({'id': img_id, 'file_name': im['fn'],
                            'width': im['w'], 'height': im['h']})
        for b, bbox in im['anns']:
            x, y, w, h = bbox
            d['annotations'].append({'id': ann_id, 'image_id': img_id,
                'category_id': b+1, 'bbox': [x, y, w, h],
                'area': w*h, 'iscrowd': 0})
            ann_id += 1
    (img_dir/'_annotations.coco.json').write_text(json.dumps(d))
    return len(d['images']), len(d['annotations'])

if Path(OUT).exists(): shutil.rmtree(OUT)
print('train:', build_split(train_items, 'train'))
print('valid:', build_split(val_items, 'valid'))
print('test :', build_split(val_items, 'test'))   # = valid (chi de thoa loader)
print('Val groups:', sorted(best))
print('COCO dataset ->', OUT)

## 3. Train RF-DETR

👉 **Chạy MỘT trong hai cell dưới** (cell còn lại thì comment/xóa):
- **Option A — RFDETRLarge** (704, Apache): chạy thoải mái trên A100-40GB.
- **Option B — RFDETR2XLarge** (880, PML 1.0): capacity tối đa; 40GB phải `batch_size=2` (chậm, dễ OOM), hợp 80GB/H100.

In [ ]:
# ===== Option A: RFDETRLarge (704, Apache) — chay thoai mai tren A100-40GB =====
from rfdetr import RFDETRLarge

# >>> Tu ngat runtime khi train xong de KHONG dot gio A100 thua <<<
AUTO_DISCONNECT = True    # False neu muon giu phien de xem checkpoint/eval tay
GRACE_SEC       = 90      # cho de ban kip Cancel truoc khi ngat

model = RFDETRLarge()                  # resolution mac dinh 704 (KHONG override)
OUT_RUN = f'{PROJECT}/rfdetr_large'

model.train(
    dataset_dir='/content/data_coco',
    epochs=60,
    batch_size=8,            # 40GB: 8-16 | 80GB/H100: 16-32
    grad_accum_steps=2,      # effective batch = 16
    lr=1e-4,
    output_dir=OUT_RUN,
    wandb=True, project='bradford-logo-rfdetr', run='rfdetr_large',
)

# ---- TU NGAT RUNTIME (chay ngay sau khi train xong) ----
# checkpoint_best_*.pth da nam tren Drive (USE_DRIVE=True) -> ngat an toan.
try:
    import wandb; wandb.finish()
except Exception:
    pass
if AUTO_DISCONNECT:
    import time
    print(f'\n[DONE] Train xong. Ngat runtime sau {GRACE_SEC}s — bam STOP/Cancel cell neu muon giu phien.')
    time.sleep(GRACE_SEC)
    from google.colab import runtime
    runtime.unassign()   # nha A100, ngung tinh gio
else:
    print('\n[DONE] Train xong. AUTO_DISCONNECT=False -> phien van mo.')

In [ ]:
# ===== Option B: RFDETR2XLarge (880, PML 1.0) — capacity toi da =====
# Yeu cau: !pip install "rfdetr[train,loggers]" rfdetr-plus  +  accept_platform_model_license=True
# 40GB: batch_size=2/grad_accum=8 (cham, de OOM). 80GB/H100: batch_size=8/grad_accum=2.
from rfdetr import RFDETR2XLarge

# >>> Tu ngat runtime khi train xong de KHONG dot gio A100 thua <<<
AUTO_DISCONNECT = True    # False neu muon giu phien de xem checkpoint/eval tay
GRACE_SEC       = 90      # cho de ban kip Cancel truoc khi ngat

model = RFDETR2XLarge(accept_platform_model_license=True)   # resolution mac dinh 880 (KHONG override)
OUT_RUN = f'{PROJECT}/rfdetr_2xlarge'

model.train(
    dataset_dir='/content/data_coco',
    epochs=60,
    batch_size=2,            # 40GB: 2 (OOM -> 1) | 80GB/H100: 8
    grad_accum_steps=8,      # effective batch = 16 (neu batch=1 -> grad_accum=16)
    lr=1e-4,
    output_dir=OUT_RUN,
    wandb=True, project='bradford-logo-rfdetr', run='rfdetr_2xlarge',
)

# ---- TU NGAT RUNTIME (chay ngay sau khi train xong) ----
# checkpoint_best_*.pth da nam tren Drive (USE_DRIVE=True) -> ngat an toan.
try:
    import wandb; wandb.finish()
except Exception:
    pass
if AUTO_DISCONNECT:
    import time
    print(f'\n[DONE] Train xong. Ngat runtime sau {GRACE_SEC}s — bam STOP/Cancel cell neu muon giu phien.')
    time.sleep(GRACE_SEC)
    from google.colab import runtime
    runtime.unassign()   # nha A100, ngung tinh gio
else:
    print('\n[DONE] Train xong. AUTO_DISCONNECT=False -> phien van mo.')

## 4. Ghi chú đánh giá

- RF-DETR **tự in mAP trên tập valid** sau mỗi epoch trong log train.
- Checkpoint tốt nhất lưu ở `OUT_RUN` (ví dụ `checkpoint_best_ema.pth` / `checkpoint_best_total.pth`).
- **So sánh công bằng với YOLO**: cùng tập val (cùng split clip-aware) → đối chiếu `mAP50-95`. Mỗi kiến trúc chạy ở resolution mặc định tối ưu của nó (RFDETRLarge 704 / 2XLarge 880 vs YOLO 1536) — không ép giống resolution.
- ⚠️ Dùng **resolution mặc định**, không override (`RFDETRLarge(resolution=...)` bản mới có bug interpolation — issue #960).

In [ ]:
# Liet ke checkpoint da luu
import os, glob
for c in sorted(glob.glob(f'{OUT_RUN}/*.pth')):
    print(round(os.path.getsize(c)/1e6, 1), 'MB ', c)